In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from data.fetcher import fetch_nse_option_chain, _generate_synthetic_chain
from data.preprocessor import clean_option_chain
from src.vol_surface import VolatilitySurface
from src.heston import heston_iv, calibrate_heston, heston_price
from src.black_scholes import bs_price

plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
df_raw = fetch_nse_option_chain('NIFTY')  # falls back to synthetic if NSE unavailable
spot = df_raw['spot'].iloc[0] if 'spot' in df_raw.columns else 22000.0
r    = df_raw['r'].iloc[0]    if 'r'    in df_raw.columns else 0.065

df = clean_option_chain(df_raw, spot=spot, r=r, min_oi=0, min_volume=0)
print(f"Clean option quotes: {len(df)}")
print(df[['strike','expiry','option_type','computed_iv','log_moneyness','T']].head(10))

In [ ]:
fig, ax = plt.subplots()
for T_val, group in df.groupby('T'):
    label = f"T = {T_val*365:.0f}d"
    ax.scatter(group['log_moneyness'], group['computed_iv'], label=label, alpha=0.6, s=20)

# BS baseline: flat vol at ATM
atm_iv = df[df['log_moneyness'].abs() < 0.02]['computed_iv'].mean()
x_range = np.linspace(-0.25, 0.25, 100)
ax.axhline(atm_iv, color='red', linestyle='--', label=f'BS flat vol ({atm_iv:.1%})')

ax.set_xlabel('Log-moneyness  ln(K/F)')
ax.set_ylabel('Implied Volatility')
ax.set_title('Volatility Smile \u2014 BS cannot explain this shape')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
surface = VolatilitySurface(df, spot=spot, r=r)

k_grid = np.linspace(-0.20, 0.20, 30)
T_grid = np.linspace(df['T'].min(), df['T'].max(), 20)
iv_grid = surface.get_iv_grid(k_grid, T_grid)

fig = go.Figure(data=[go.Surface(
    x=k_grid,
    y=T_grid * 365,  # display in days
    z=iv_grid,
    colorscale='Viridis',
    colorbar=dict(title='Implied Vol'),
)])
fig.update_layout(
    title='Implied Volatility Surface',
    scene=dict(
        xaxis_title='Log-moneyness ln(K/F)',
        yaxis_title='Days to Expiry',
        zaxis_title='Implied Volatility',
    ),
    width=900, height=600
)
fig.show()

In [ ]:
T_near = df['T'].min()
slice_df = df[df['T'] == T_near].copy()

params, rmse = calibrate_heston(
    slice_df['log_moneyness'].values,
    slice_df['computed_iv'].values,
    T_near, spot, r
)
print("Calibrated Heston parameters:")
for k, v in params.items():
    print(f"  {k:8s} = {v:.4f}")
print(f"\nCalibration RMSE: {rmse*100:.2f} vol points")

In [ ]:
k_range = np.linspace(-0.25, 0.25, 50)
F = spot * np.exp(r * T_near)
strikes_range = F * np.exp(k_range)
heston_ivs = [heston_iv(spot, K_i, T_near, r, **params) for K_i in strikes_range]

fig, ax = plt.subplots()
ax.scatter(slice_df['log_moneyness'], slice_df['computed_iv'],
           label='Market IV', color='navy', s=30, zorder=5)
ax.plot(k_range, heston_ivs, label='Heston fit', color='crimson', linewidth=2)
ax.axhline(np.sqrt(params['theta']), color='gray', linestyle='--',
           label=f"Heston long-run vol = {np.sqrt(params['theta']):.1%}")

ax.set_xlabel('Log-moneyness  ln(K/F)')
ax.set_ylabel('Implied Volatility')
ax.set_title(f'Heston vs Market \u2014 T = {T_near*365:.0f} days  |  RMSE = {rmse*100:.2f}%')
ax.legend()
plt.tight_layout()
plt.show()